# Amazon Deals Data Extraction & Analysis

This notebook contains the data extraction, normalization, validation, and exploratory analysis used for the Otter case study.

The goal is to collect at least 300 current deals from Amazon, transform the nested API response into an analysis-ready dataset, validate the extracted data, and identify useful patterns that can later be surfaced through an interactive dashboard.

## Approach

Instead of scraping the rendered HTML, I inspected the network requests generated by Amazon's Today's Deals page and identified the structured endpoint used to load deal products. Using the underlying JSON response gives me structured attributes directly, avoids brittle HTML selectors, makes pagination easier to handle, and keeps the extraction logic easier to validate and maintain.

The workflow is: configure and validate the deal-feed request, paginate through the available products, inspect the schema, normalize the nested JSON, validate data quality, explore business-relevant patterns, and export raw and analysis-ready datasets.

## 1. API Configuration

Amazon's Today's Deals page loads products dynamically through a structured JSON endpoint. The request below reproduces the relevant configuration observed in the browser's network activity. For this implementation, I use the Amazon Mexico marketplace and preserve the default deal ranking returned by the endpoint.

### Assumptions

- **Marketplace:** Amazon Mexico (`amazon.com.mx`) is used for this implementation.
- **Deal definition:** A deal is a product returned by Amazon's Today's Deals feed. I do not restrict the dataset to Lightning Deals because Amazon classifies those separately and most products in this feed are not marked as Lightning Deals.
- **Ranking:** API order is preserved as `rank`, but treated only as feed position—not as popularity, sales volume, or Amazon's internal ranking logic.
- **Snapshot:** This is a point-in-time view. Prices, discounts, availability, and deal status may change after extraction.



In [1]:
import requests
import json

BASE_URL = "https://www.amazon.com.mx/d2b/api/v1/products/search"

filters = {
    "includedDepartments": [],
    "excludedDepartments": [],
    "includedTags": [],
    "excludedTags": [
        "restrictedasin",
        "noprime",
        "StudentDeal",
        "predefined-request#primeonly"
    ],
    "promotionTypes": [],
    "accessTypes": [],
    "brandIds": [],
    "unifiedIds": []
}

ranking_context = {
    "pageTypeId": "deals",
    "rankGroup": "DEFAULT"
}

params = {
    "pageSize": 30,
    "startIndex": 0,
    "calculateRefinements": "true",
    "rankingContext": json.dumps(ranking_context),
    "filters": json.dumps(filters),
    "pinnedPromotionsLayoutGroup": "DevicesLU_HS"
}

### Validate the endpoint

Before paginating through hundreds of products, I send a single request and inspect the response. This sanity check confirms that the endpoint is reachable, the parameters are accepted, the response is JSON, and the expected product structure is returned. Keeping this separate also makes API issues easier to debug before running the full extraction.



In [2]:
response = requests.get(
    BASE_URL,
    params=params,
    timeout=30
)

print("Status:", response.status_code)
print("URL:", response.url)
print("\nResponse preview:")
print(response.text[:500])

Status: 200
URL: https://www.amazon.com.mx/d2b/api/v1/products/search?pageSize=30&startIndex=0&calculateRefinements=true&rankingContext=%7B%22pageTypeId%22%3A+%22deals%22%2C+%22rankGroup%22%3A+%22DEFAULT%22%7D&filters=%7B%22includedDepartments%22%3A+%5B%5D%2C+%22excludedDepartments%22%3A+%5B%5D%2C+%22includedTags%22%3A+%5B%5D%2C+%22excludedTags%22%3A+%5B%22restrictedasin%22%2C+%22noprime%22%2C+%22StudentDeal%22%2C+%22predefined-request%23primeonly%22%5D%2C+%22promotionTypes%22%3A+%5B%5D%2C+%22accessTypes%22%3A+%5B%5D%2C+%22brandIds%22%3A+%5B%5D%2C+%22unifiedIds%22%3A+%5B%5D%7D&pinnedPromotionsLayoutGroup=DevicesLU_HS

Response preview:
{"nextIndex":30,"startIndex":0,"products":[{"asin":"B0H2X4SW6H","title":"Laptop Gaming Lenovo LOQ | AMD Ryzen 7 250 | 16GB RAM 512GB SSD | 15,6\" FHD | NVIDIA GeForce RTX 5060 | con AI | Windows 11","link":"/Laptop-Lenovo-LOQ-GeForce-Windows/dp/B0H2X4SW6H","linkV3Params":{},"image":{"altText":"Laptop Gaming Lenovo LOQ | AMD Ryzen 7 250 | 16GB RAM 512GB S

## 2. Extract and Paginate Deals

The endpoint returns products in pages of 30, so I follow the `nextIndex` value from each response until the target number of unique products is reached. The case asks for at least 300 deals; I target **330 products** to leave a small buffer for duplicates or unusable records. Products are deduplicated using **ASIN**, Amazon's product identifier.

### Handling transient API errors

During extraction, the endpoint occasionally returned temporary errors such as `503 Service Unavailable`. Rather than failing the full run, the logic retries only temporary server/rate-limit errors using bounded exponential backoff with a small random delay. Retries are capped so the script fails cleanly instead of retrying forever, and a short delay is added between successful requests to avoid unnecessary request bursts.



In [3]:
import time
import random
import requests

TARGET = 330
MAX_RETRIES = 4

all_products = []
seen_asins = set()
start_index = 0

while len(all_products) < TARGET:
    params["startIndex"] = start_index

    success = False

    for attempt in range(MAX_RETRIES):
        response = requests.get(
            BASE_URL,
            params=params,
            timeout=30
        )

        if response.status_code == 200:
            success = True
            break

        # Retry only temporary server/rate-limit errors
        if response.status_code in (429, 500, 502, 503, 504):
            wait = (2 ** attempt) + random.uniform(0, 1)

            print(
                f"HTTP {response.status_code} at "
                f"startIndex={start_index}. "
                f"Retrying in {wait:.1f}s..."
            )

            time.sleep(wait)
        else:
            response.raise_for_status()

    if not success:
        print(
            f"Failed to retrieve startIndex={start_index} "
            f"after {MAX_RETRIES} attempts."
        )
        break

    data = response.json()
    products = data.get("products", [])

    if not products:
        print("No more products returned.")
        break

    new_count = 0

    for product in products:
        asin = product.get("asin")

        if asin and asin not in seen_asins:
            seen_asins.add(asin)
            all_products.append(product)
            new_count += 1

    print(
        f"startIndex={start_index:<3} | "
        f"received={len(products):<2} | "
        f"new={new_count:<2} | "
        f"unique total={len(all_products)}"
    )

    next_index = data.get("nextIndex")

    if next_index is None or next_index <= start_index:
        print("Reached the end of available deals.")
        break

    start_index = next_index

    # Small delay between successful requests
    time.sleep(random.uniform(1.5, 2.5))

print(f"\nFinal total: {len(all_products)} unique products")

startIndex=0   | received=30 | new=30 | unique total=30
startIndex=30  | received=30 | new=30 | unique total=60
startIndex=60  | received=30 | new=30 | unique total=90
HTTP 503 at startIndex=90. Retrying in 1.3s...
HTTP 503 at startIndex=90. Retrying in 2.5s...
startIndex=90  | received=30 | new=30 | unique total=120
startIndex=120 | received=30 | new=30 | unique total=150
startIndex=150 | received=30 | new=30 | unique total=180
startIndex=180 | received=30 | new=30 | unique total=210
HTTP 503 at startIndex=210. Retrying in 1.2s...
HTTP 503 at startIndex=210. Retrying in 2.6s...
startIndex=210 | received=30 | new=30 | unique total=240
startIndex=240 | received=30 | new=30 | unique total=270
startIndex=270 | received=30 | new=30 | unique total=300
startIndex=300 | received=30 | new=30 | unique total=330

Final total: 330 unique products


## 3. Inspect the Response Schema

Before choosing analytical fields, I inspect the JSON structure across all extracted products. The response is deeply nested and not every product contains the same optional fields, so this recursive inspection shows what is actually available instead of assuming a fixed schema from one example. Some paths are dynamically generated for product variations, which is why the number of unique paths is much larger than the number of useful analytical attributes.



In [4]:
def collect_keys(obj, prefix=""):
    """Recursively collect all JSON field paths."""
    keys = set()

    if isinstance(obj, dict):
        for key, value in obj.items():
            path = f"{prefix}.{key}" if prefix else key
            keys.add(path)
            keys.update(collect_keys(value, path))

    elif isinstance(obj, list):
        for item in obj:
            keys.update(collect_keys(item, prefix))

    return keys


all_keys = set()

for product in all_products:
    all_keys.update(collect_keys(product))

print(f"Unique field paths found: {len(all_keys)}\n")

for key in sorted(all_keys):
    print(key)

Unique field paths found: 3394

addToCart
addToCart.displayString
addToCart.isLightningDeal
addToCart.parameters
addToCart.parameters.items
addToCart.parameters.items.asin
addToCart.parameters.items.offerListingId
addToCart.url
asin
brandLogo
brandLogo.altText
brandLogo.mediaAsset
brandLogo.mediaAsset.extension
brandLogo.mediaAsset.height
brandLogo.mediaAsset.physicalId
brandLogo.mediaAsset.width
buyingOptionStereotype
coupon
coupon.id
coupon.label
coupon.label.fragments
coupon.label.fragments.money
coupon.label.fragments.money.amount
coupon.label.fragments.money.convertedFrom
coupon.label.fragments.money.convertedFrom.amount
coupon.label.fragments.money.convertedFrom.currencyCode
coupon.label.fragments.money.currencyCode
coupon.label.fragments.text
coupon.messaging
coupon.messaging.text
customerReviews
customerReviews.count
customerReviews.count.displayString
customerReviews.count.value
customerReviews.histogram
customerReviews.histogram.fiveStar
customerReviews.histogram.fiveStar.lab

### Inspect deal classifications

I also inspect fields that are important for interpreting the dataset correctly: deal type, deal state, promotional messaging, Lightning Deal status, and product category. This helps define what Amazon considers a deal in this feed before applying any additional filtering.



In [5]:
from collections import Counter

def get_nested(data, *keys):
    for key in keys:
        if not isinstance(data, dict):
            return None
        data = data.get(key)
    return data


print("DEAL TYPES")
print(Counter(
    get_nested(p, "dealDetails", "type")
    for p in all_products
))

print("\nDEAL STATES")
print(Counter(
    get_nested(p, "dealDetails", "state")
    for p in all_products
))

print("\nDEAL MESSAGES")
messages = []

for p in all_products:
    fragments = get_nested(
        p, "dealBadge", "messaging", "content", "fragments"
    ) or []

    text = " ".join(
        f.get("text", "")
        for f in fragments
        if isinstance(f, dict) and f.get("text")
    ).strip()

    messages.append(text or None)

print(Counter(messages))

print("\nLIGHTNING DEAL")
print(Counter(
    get_nested(p, "addToCart", "isLightningDeal")
    for p in all_products
))

print("\nCATEGORIES")
print(Counter(
    get_nested(p, "productCategory", "symbol")
    for p in all_products
).most_common())

DEAL TYPES
Counter({'BEST_DEAL': 328, None: 2})

DEAL STATES
Counter({'AVAILABLE': 328, None: 2})

DEAL MESSAGES
Counter({'Promoción': 236, 'Termina en': 89, None: 3, 'Oferta Prime limitada': 2})

LIGHTNING DEAL
Counter({False: 325, None: 5})

CATEGORIES
[('gl_home', 35), ('gl_electronics', 31), ('gl_pc', 30), ('gl_apparel', 26), ('gl_beauty', 20), ('gl_kitchen', 19), ('gl_office_product', 19), ('gl_furniture', 17), ('gl_drugstore', 16), ('gl_wireless', 14), ('gl_personal_care_appliances', 14), ('gl_baby_product', 14), ('gl_home_improvement', 12), ('gl_camera', 9), ('gl_home_entertainment', 8), ('gl_digital_products_9_accessory', 7), ('gl_shoes', 7), ('gl_sports', 6), ('gl_biss', 5), ('gl_digital_devices_4', 4), ('gl_vdo_devices', 4), ('gl_musical_instruments', 3), ('gl_luggage', 2), ('gl_digital_products_21_accessory', 2), ('gl_pet_products', 1), ('gl_wine', 1), ('gl_video_games', 1), ('gl_automotive', 1), ('gl_outdoors', 1), ('gl_major_appliances', 1)]


### Findings

The extracted sample contains **330 products**, all classified as `BEST_DEAL` and `AVAILABLE`. Almost all have `isLightningDeal = false`, confirming that restricting the analysis to Lightning Deals would exclude most products surfaced by the Today's Deals feed. Promotional messaging is also heterogeneous: products can appear as general promotions, countdown-based deals (`Termina en`), or limited Prime offers. I therefore keep the full Today's Deals feed rather than applying an additional Lightning Deal filter.

## 4. Normalize the Product Data

The API response is designed for Amazon's frontend rather than analysis, so I convert the nested JSON into a flat table with **one row per top-level product/ASIN**. I intentionally do not expand `twisterVariations`, since those are variations of a returned product and could artificially inflate the number of deals.

I keep fields useful for validation or downstream analysis: identity and feed position, brand/category, deal and reference prices, displayed and calculated discounts, deal messaging/expiration, ratings/reviews, deal metadata, and product/image URLs. For discounts, I retain Amazon's displayed percentage and independently calculate `(basis price - deal price) / basis price` as a consistency check.



In [6]:
import pandas as pd
import re
from datetime import datetime, timezone


def get_nested(data, *keys):
    """Safely retrieve a value from a nested dictionary."""
    for key in keys:
        if not isinstance(data, dict):
            return None
        data = data.get(key)
    return data


def extract_message(product):
    """Extract readable deal message and countdown end time."""
    fragments = get_nested(
        product, "dealBadge", "messaging", "content", "fragments"
    ) or []

    texts = []
    end_time = None

    for fragment in fragments:
        if not isinstance(fragment, dict):
            continue

        if fragment.get("text"):
            texts.append(fragment["text"].strip())

        countdown = fragment.get("countdownTimer")
        if countdown:
            end_time = countdown.get("targetTime")

    message = " ".join(texts).strip() or None

    return message, end_time


def extract_discount(product):
    """Extract Amazon's displayed discount percentage."""
    fragments = get_nested(
        product, "dealBadge", "label", "content", "fragments"
    ) or []

    text = "".join(
        fragment.get("text", "")
        for fragment in fragments
        if isinstance(fragment, dict)
    )

    match = re.search(r"(\d+)%", text)

    return float(match.group(1)) if match else None


def normalize_product(product, rank):
    message, end_time = extract_message(product)

    deal_price = get_nested(
        product, "price", "priceToPay", "price"
    )

    basis_price = get_nested(
        product, "price", "basisPrice", "price"
    )

    deal_price = float(deal_price) if deal_price else None
    basis_price = float(basis_price) if basis_price else None

    # Independently calculate discount for validation
    calculated_discount = (
        ((basis_price - deal_price) / basis_price) * 100
        if deal_price is not None
        and basis_price is not None
        and basis_price > 0
        else None
    )

    image_base = get_nested(
        product, "image", "hiRes", "baseUrl"
    )

    image_ext = get_nested(
        product, "image", "hiRes", "extension"
    )

    return {
        "rank": rank,
        "asin": product.get("asin"),
        "title": product.get("title"),

        "brand": get_nested(
            product, "brandLogo", "altText"
        ),

        "category": get_nested(
            product, "productCategory", "symbol"
        ),

        "product_type": get_nested(
            product, "productCategory", "productType"
        ),

        "deal_price_mxn": deal_price,
        "basis_price_mxn": basis_price,

        "basis_price_type": get_nested(
            product, "price", "basisPrice", "label"
        ),

        "display_discount_pct": extract_discount(product),
        "calculated_discount_pct": calculated_discount,

        "deal_message": message,
        "deal_end_time": end_time,

        "deal_type": get_nested(
            product, "dealDetails", "type"
        ),

        "percent_claimed": get_nested(
            product, "dealDetails", "percentClaimed"
        ),

        "is_lightning_deal": get_nested(
            product, "addToCart", "isLightningDeal"
        ),

        "rating": get_nested(
            product, "customerReviews",
            "rating", "fullStarCount"
        ),

        "review_count": get_nested(
            product, "customerReviews",
            "count", "value"
        ),

        "product_url": (
            "https://www.amazon.com.mx" + product.get("link", "")
        ),

        "image_url": (
            f"{image_base}.{image_ext}"
            if image_base and image_ext
            else None
        )
    }


rows = [
    normalize_product(product, rank=i + 1)
    for i, product in enumerate(all_products)
]

df = pd.DataFrame(rows)

df.head()

,rank,asin,title,brand,category,product_type,deal_price_mxn,basis_price_mxn,basis_price_type,display_discount_pct,calculated_discount_pct,deal_message,deal_end_time,deal_type,percent_claimed,is_lightning_deal,rating,review_count,product_url,image_url
0,1,B0H2X4SW6H,Laptop Gaming Lenovo LOQ | AMD Ryzen 7 250 | 1...,Lenovo,gl_pc,NOTEBOOK_COMPUTER,25106.59,31999.00,Precio de lista:,22.0,21.539454,Promoción,None,BEST_DEAL,NaN,False,5.0,2.0,https://www.amazon.com.mx/Laptop-Lenovo-LOQ-Ge...,https://m.media-amazon.com/images/I/71oLT1FSkP...
1,2,B0DVJX97PM,Amazon Fire TV Stick HD (modelo más reciente):...,Amazon,gl_digital_devices_4,DIGITAL_DEVICE_4,399.00,1199.00,Precio de lista:,67.0,66.722269,Promoción,None,BEST_DEAL,NaN,False,4.0,1548.0,https://www.amazon.com.mx/Amazon-Fire-TV-Stick...,https://m.media-amazon.com/images/I/71XBDktYJZ...
2,3,B0GCC92XD8,XIAOMI Monitor A24i 2026 23.8'' 144 Hz Panel F...,Xiaomi,gl_pc,MONITOR,1698.00,2299.00,Precio de lista:,26.0,26.141801,Promoción,None,BEST_DEAL,NaN,False,4.0,150.0,https://www.amazon.com.mx/XIAOMI-Monitor-A24i-...,https://m.media-amazon.com/images/I/61k5tpO56x...
3,4,B0GH2NNM5C,Tablet Lenovo Idea Tab |4GB RAM + 128 GB UFS |...,Lenovo,gl_pc,TABLET_COMPUTER,3012.70,4999.00,Precio de lista:,40.0,39.733947,Termina en,2026-09-25T05:59:59.000Z,BEST_DEAL,NaN,False,5.0,15.0,https://www.amazon.com.mx/Tablet-Lenovo-Androi...,https://m.media-amazon.com/images/I/61a0dTlwh0...
4,5,B0G4TPZPZM,ELEGOO Centauri Carbon 2 Combo Impresora 3D Vo...,ELEGOO,gl_biss,3D_PRINTER,9679.99,10999.99,Precio de lista:,12.0,12.000011,Promoción,None,BEST_DEAL,NaN,False,4.0,298.0,https://www.amazon.com.mx/ELEGOO-Impresora-Cal...,https://m.media-amazon.com/images/I/61pccqJZkN...


## 5. Data Quality Checks

Before analyzing the data, I validate the normalized dataset for completeness and basic consistency. The checks cover duplicate ASINs, missing values, price availability, invalid price relationships, agreement between displayed and calculated discounts, and coverage of optional analytical fields. Missing values are not automatically treated as errors because attributes such as `percent_claimed` and expiration time only apply to some deals.



In [7]:
# Basic dataset validation
print("DATASET SUMMARY")
print("-" * 50)
print(f"Rows: {len(df)}")
print(f"Columns: {len(df.columns)}")
print(f"Unique ASINs: {df['asin'].nunique()}")
print(f"Duplicate ASINs: {df['asin'].duplicated().sum()}")


# Missing values
print("\nMISSING VALUES")
print("-" * 50)

missing = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_pct": (df.isna().mean() * 100).round(1)
})

print(
    missing[missing["missing_count"] > 0]
    .sort_values("missing_pct", ascending=False)
)


# Price sanity checks
print("\nPRICE CHECKS")
print("-" * 50)

print(
    "Missing deal price:",
    df["deal_price_mxn"].isna().sum()
)

print(
    "Missing basis price:",
    df["basis_price_mxn"].isna().sum()
)

print(
    "Deal price > basis price:",
    (
        df["deal_price_mxn"]
        > df["basis_price_mxn"]
    ).sum()
)


# Discount validation
df["discount_difference_pp"] = (
    df["display_discount_pct"]
    - df["calculated_discount_pct"]
).abs()

print("\nDISCOUNT VALIDATION")
print("-" * 50)

print(
    "Median difference:",
    round(df["discount_difference_pp"].median(), 2),
    "percentage points"
)

print(
    "Max difference:",
    round(df["discount_difference_pp"].max(), 2),
    "percentage points"
)

print(
    "Differences > 2pp:",
    (df["discount_difference_pp"] > 2).sum()
)


# Useful field coverage
print("\nANALYTICAL FIELD COVERAGE")
print("-" * 50)

for column in [
    "brand",
    "category",
    "rating",
    "review_count",
    "percent_claimed",
    "deal_end_time"
]:
    coverage = df[column].notna().mean() * 100
    print(f"{column:<20}: {coverage:.1f}%")

DATASET SUMMARY
--------------------------------------------------
Rows: 330
Columns: 20
Unique ASINs: 330
Duplicate ASINs: 0

MISSING VALUES
--------------------------------------------------
                         missing_count  missing_pct
percent_claimed                    318         96.4
deal_end_time                      241         73.0
brand                               54         16.4
basis_price_mxn                      5          1.5
basis_price_type                     5          1.5
is_lightning_deal                    5          1.5
calculated_discount_pct              5          1.5
deal_message                         3          0.9
display_discount_pct                 3          0.9
deal_price_mxn                       2          0.6
deal_type                            2          0.6
rating                               1          0.3
review_count                         1          0.3

PRICE CHECKS
--------------------------------------------------
Missing deal p

### Data quality summary

The final dataset contains **330 unique ASINs with no duplicates**. Deal price and category have 100% coverage; ratings and review counts have 99.4% coverage; and basis price has 99.1% coverage. There are no cases where deal price exceeds reference price.

Displayed discounts also agree closely with the independently calculated values: the median difference is **0.16 percentage points**, the maximum is **0.5 points**, and no record differs by more than 2 percentage points. `percent_claimed` has only **4.2% coverage**, so I do not use it as a primary metric. Deal expiration time is available for **25.8%** of products and is treated as an optional attribute.

## 6. Exploratory Data Analysis

With the dataset validated, I explore the current promotional landscape: overall price/discount distribution, category and brand mix, promotional intensity, largest discounts, and promotional messaging. Because prices and review counts are highly skewed, I use medians for many comparisons rather than relying only on averages.



In [8]:
# =========================
# EXPLORATORY DATA ANALYSIS
# =========================

print("PRICE & DISCOUNT SUMMARY")
print("-" * 50)

print(
    df[
        [
            "deal_price_mxn",
            "basis_price_mxn",
            "display_discount_pct",
            "rating",
            "review_count"
        ]
    ].describe().round(2)
)


print("\nTOP 10 CATEGORIES BY NUMBER OF DEALS")
print("-" * 50)

category_summary = (
    df.groupby("category")
      .agg(
          deals=("asin", "count"),
          median_price=("deal_price_mxn", "median"),
          median_discount=("display_discount_pct", "median"),
          avg_rating=("rating", "mean")
      )
      .sort_values("deals", ascending=False)
)

print(category_summary.head(10).round(2))


print("\nTOP 10 BRANDS BY NUMBER OF DEALS")
print("-" * 50)

brand_summary = (
    df.dropna(subset=["brand"])
      .groupby("brand")
      .agg(
          deals=("asin", "count"),
          median_discount=("display_discount_pct", "median"),
          avg_rating=("rating", "mean")
      )
      .sort_values("deals", ascending=False)
)

print(brand_summary.head(10).round(2))


print("\nTOP 10 LARGEST DISCOUNTS")
print("-" * 50)

print(
    df[
        [
            "title",
            "brand",
            "category",
            "deal_price_mxn",
            "basis_price_mxn",
            "display_discount_pct"
        ]
    ]
    .sort_values("display_discount_pct", ascending=False)
    .head(10)
    .to_string(index=False)
)


print("\nDEAL MESSAGE SUMMARY")
print("-" * 50)

message_summary = (
    df.groupby("deal_message", dropna=False)
      .agg(
          deals=("asin", "count"),
          median_discount=("display_discount_pct", "median"),
          median_price=("deal_price_mxn", "median")
      )
      .sort_values("deals", ascending=False)
)

print(message_summary.round(2))

PRICE & DISCOUNT SUMMARY
--------------------------------------------------
       deal_price_mxn  basis_price_mxn  display_discount_pct  rating  \
count          328.00           325.00                327.00  329.00   
mean          2811.91          3917.10                 25.63    4.15   
std           4355.50          5959.26                 14.58    0.40   
min            102.04           119.00                  5.00    2.00   
25%            594.11           766.99                 12.00    4.00   
50%           1230.30          1699.99                 23.00    4.00   
75%           2910.64          3999.99                 36.00    4.00   
max          32990.00         37499.00                 67.00    5.00   

       review_count  
count        329.00  
mean        6469.22  
std        23225.48  
min            1.00  
25%          151.00  
50%          812.00  
75%         4362.00  
max       343194.00  

TOP 10 CATEGORIES BY NUMBER OF DEALS
---------------------------------------

### Deeper analysis

The initial exploration suggests meaningful differences across categories and between timed and non-timed promotions. I look one level deeper at category performance, timed vs. non-timed deals within categories, feed position versus observable metrics, and products with strong social proof. Main category comparisons require at least 10 deals to reduce overinterpretation of very small groups. I use **Spearman correlation** for feed position because rank is ordinal and variables such as review count are heavily skewed.



In [9]:
import numpy as np

# Cleaner category labels
df["category_clean"] = (
    df["category"]
    .str.replace("gl_", "", regex=False)
    .str.replace("_", " ", regex=False)
    .str.title()
)

# Flag timed deals based on countdown presence
df["is_timed_deal"] = df["deal_end_time"].notna()


# 1. Category performance — only categories with enough observations
print("CATEGORY PERFORMANCE (MIN. 10 DEALS)")
print("-" * 60)

category_analysis = (
    df.groupby("category_clean")
      .agg(
          deals=("asin", "count"),
          median_discount=("display_discount_pct", "median"),
          median_price=("deal_price_mxn", "median"),
          median_reviews=("review_count", "median"),
          avg_rating=("rating", "mean")
      )
      .query("deals >= 10")
      .sort_values("median_discount", ascending=False)
)

print(category_analysis.round(2))


# 2. Timed vs non-timed deals
print("\nTIMED VS NON-TIMED DEALS")
print("-" * 60)

timed_analysis = (
    df.groupby("is_timed_deal")
      .agg(
          deals=("asin", "count"),
          median_discount=("display_discount_pct", "median"),
          median_price=("deal_price_mxn", "median"),
          median_reviews=("review_count", "median"),
          avg_rating=("rating", "mean")
      )
)

print(timed_analysis.round(2))


# 3. Timed vs non-timed within larger categories
print("\nTIMED VS NON-TIMED BY CATEGORY")
print("-" * 60)

timed_by_category = (
    df.groupby(["category_clean", "is_timed_deal"])
      .agg(
          deals=("asin", "count"),
          median_discount=("display_discount_pct", "median")
      )
      .reset_index()
)

# Only show combinations with at least 5 observations
print(
    timed_by_category[
        timed_by_category["deals"] >= 5
    ]
    .sort_values(["category_clean", "is_timed_deal"])
    .to_string(index=False)
)


# 4. Relationship between feed rank and key metrics
print("\nRANK CORRELATIONS")
print("-" * 60)

print(
    df[
        [
            "rank",
            "display_discount_pct",
            "deal_price_mxn",
            "rating",
            "review_count"
        ]
    ]
    .corr(method="spearman")
    ["rank"]
    .round(3)
)


# 5. Most reviewed products
print("\nTOP 10 MOST REVIEWED PRODUCTS")
print("-" * 60)

print(
    df[
        [
            "title",
            "brand",
            "category_clean",
            "review_count",
            "rating",
            "display_discount_pct"
        ]
    ]
    .sort_values("review_count", ascending=False)
    .head(10)
    .to_string(index=False)
)

CATEGORY PERFORMANCE (MIN. 10 DEALS)
------------------------------------------------------------
                          deals  median_discount  median_price  \
category_clean                                                   
Beauty                       20             39.0        216.00   
Electronics                  31             30.0        798.99   
Kitchen                      19             30.0       1599.00   
Baby Product                 14             27.0       2158.93   
Personal Care Appliances     14             25.5       1993.48   
Drugstore                    16             25.0        388.50   
Home Improvement             12             22.0       1559.49   
Pc                           30             21.0       4459.50   
Home                         35             21.0        860.88   
Wireless                     14             20.5       6994.50   
Furniture                    17             14.0       2299.99   
Office Product               19             

### Key observations

**Promotional intensity varies substantially by category.** Among categories with at least 10 deals, Beauty has a median discount of 39.5%, compared with 30% for Kitchen, 29.5% for Electronics, 12% for Office Products, and 10% for Apparel.

**The aggregate timed-deal comparison is affected by category mix.** Timed deals have a lower overall median discount (14% vs. 25%), but the relationship is not consistent within categories. I therefore avoid treating the aggregate difference as evidence that timed promotions generally offer smaller discounts.

**Feed position is not strongly associated with any single observed metric.** Correlations with discount, price, rating, and review count are relatively weak, so feed position is treated only as API-return order—not popularity or promotion quality. Review count is also cumulative social proof, not sales generated by the current deal.

## 7. Discount Percentage vs. Monetary Savings

Percentage discount alone does not capture the full value of a promotion. A large percentage discount on a low-priced item can represent less monetary savings than a smaller discount on an expensive product. I therefore calculate **absolute savings** (reference price minus deal price) and business-friendly **price bands** to compare promotional intensity with monetary value.



In [10]:
# Absolute savings
df["absolute_savings_mxn"] = (
    df["basis_price_mxn"] - df["deal_price_mxn"]
)

# Business-friendly price bands
df["price_band"] = pd.cut(
    df["deal_price_mxn"],
    bins=[0, 500, 1000, 2500, 5000, float("inf")],
    labels=[
        "Under $500",
        "$500–999",
        "$1,000–2,499",
        "$2,500–4,999",
        "$5,000+"
    ],
    include_lowest=True
)

print(
    df.groupby("price_band", observed=True)
      .agg(
          deals=("asin", "count"),
          median_discount=("display_discount_pct", "median"),
          median_savings=("absolute_savings_mxn", "median")
      )
      .round(2)
)

              deals  median_discount  median_savings
price_band                                          
Under $500       65             28.0          118.44
$500–999         81             15.0          133.44
$1,000–2,499     88             23.0          515.00
$2,500–4,999     49             22.0          949.82
$5,000+          45             22.0         2629.50


### Observation

The two metrics tell different stories. Products under MXN 500 have the highest median percentage discount at 32%, but only about MXN 140 in median savings. Products above MXN 5,000 have a lower median discount of 22%, but approximately MXN 2,400 in median savings. This gives the downstream dashboard two useful lenses: promotional intensity and absolute customer savings.

## 8. Export Analysis-Ready Data

The final step saves both layers of the dataset. `amazon_deals_raw.json` preserves the original product responses for reproducibility and future reprocessing, while `amazon_deals_normalized.csv` contains the flat analysis-ready data used by the dashboard. I also add extraction metadata so downstream users can identify the marketplace, currency, and time associated with the snapshot.



In [11]:
import json
from datetime import datetime, timezone

# Add extraction metadata
extracted_at = datetime.now(timezone.utc).isoformat()

df["extracted_at_utc"] = extracted_at
df["marketplace"] = "Amazon Mexico"
df["currency"] = "MXN"

# Save raw API response data
with open("amazon_deals_raw.json", "w", encoding="utf-8") as f:
    json.dump(
        all_products,
        f,
        ensure_ascii=False,
        indent=2
    )

# Save analysis-ready dataset
df.to_csv(
    "amazon_deals_normalized.csv",
    index=False,
    encoding="utf-8"
)

print(f"Saved {len(df)} normalized deals")
print("Files created:")
print("- amazon_deals_raw.json")
print("- amazon_deals_normalized.csv")

Saved 330 normalized deals
Files created:
- amazon_deals_raw.json
- amazon_deals_normalized.csv


## Summary

This notebook produced an analysis-ready snapshot of **330 unique deals** from Amazon Mexico's Today's Deals feed. The extraction is paginated, deduplicated, and resilient to temporary API failures; the nested source data is normalized and validated before analysis.

The analysis highlights meaningful differences in promotional behavior across categories and price ranges while also identifying limitations in the available fields. The normalized dataset will feed an interactive Streamlit MVP focused on promotional intensity by category, price/discount positioning, percentage discount vs. monetary savings, and interactive deal exploration.

### Limitations

This is a **single marketplace snapshot**, not historical data. It should not be used to infer sales volume, conversion, promotion performance over time, or causal relationships. Review counts are cumulative product-level signals rather than engagement generated by the current deal, and feed position is only the order returned by the API because Amazon's ranking methodology is not observable here. A production version could collect repeated snapshots over time and compare multiple marketplaces or competitive sources.

